# Safe Bayesian Optimization 利用ガイド

このNotebookでは、**安全制約付きベイズ最適化（Safe BO）**の基本的な使い方を日本語で説明します。

実験で安全制約を破りたくない場合に、GPの予測平均だけでなく不確実性も考慮して次の候補を選びます。実際の安全性を保証するものではありません。

## このNotebookで学ぶこと

1. 安全制約と初期安全観測を設定する
2. BoTorchのGPを学習する
3. `SafeBOWorkflow.ask()` で候補を提案する
4. 実験結果を `tell()` で登録する
5. GPを更新し、次の候補を提案する
6. `mode="expansion"` に切り替えて安全領域内を探索する

**対象範囲:** 連続変数、連続値の安全制約、逐次提案 `q=1`。

In [ ]:
import torch
from botorch.acquisition.analytic import PosteriorMean
from botorch.fit import fit_gpytorch_mll
from botorch.models import SingleTaskGP
from gpytorch.mlls import ExactMarginalLogLikelihood

from robotorchan.optim import SafeBOWorkflow
from robotorchan.semantics import ContinuousConstraint, SafetySpecification

torch.set_default_dtype(torch.double)
bounds = torch.tensor([[0.0], [1.0]])


def objective(x):
    return -(x - 0.3).square()


def safety_output(x):
    return x - 0.7


def fit_gp(x, y):
    model = SingleTaskGP(x, y)
    fit_gpytorch_mll(ExactMarginalLogLikelihood(model.likelihood, model))
    model.eval()
    return model


## 1. 安全制約と初期観測

安全出力 `g(x) <= 0` を要求します。`ContinuousConstraint(0, 0.0)` は安全モデルの出力インデックス0に対する上限制約です。

`SafetySpecification` は**実測値が安全な初期点**を要求します。安全性の検証をGPの予測だけに委ねないことが重要です。

In [ ]:
initial_x = torch.tensor([[0.05], [0.10], [0.15]])
initial_g = safety_output(initial_x)
spec = SafetySpecification(
    constraints=(ContinuousConstraint(0, 0.0),),
    initial_X=initial_x,
    observed_constraints=initial_g,
    beta=4.0,
)
workflow = SafeBOWorkflow(spec, bounds)
train_x = initial_x.clone()
train_y = objective(train_x)
train_g = initial_g.clone()
print("初期安全観測:", train_x.squeeze(-1).tolist())


## 2. GPの学習と安全候補の提案

目的関数と安全制約に、それぞれ `SingleTaskGP` を使用します。獲得関数には説明を簡単にするため `PosteriorMean` を用います。

`ask()` は信頼区間で安全と判定された候補だけを返します。安全候補が見つからない場合は例外になります。

In [ ]:
objective_model = fit_gp(train_x, train_y)
safety_model = fit_gp(train_x, train_g)
acquisition = PosteriorMean(objective_model)
candidate = workflow.ask(
    safety_model,
    mode="objective",
    acq_function=acquisition,
    num_samples=256,
    seed=10,
)
print("提案候補:", candidate.item())


## 3. 実験結果を登録する

`tell()` に渡すのは**実測した安全制約値**です。実際に危険な結果が得られた場合も記録されます。目的関数値は、呼び出し側で管理します。

In [ ]:
observed_y = objective(candidate)
observed_g = safety_output(candidate)
workflow.tell(candidate, observed_g)
train_x = torch.cat([train_x, candidate], dim=0)
train_y = torch.cat([train_y, observed_y], dim=0)
train_g = torch.cat([train_g, observed_g], dim=0)
print("実測安全制約:", observed_g.item())
print("記録した観測数:", workflow.observed_X.shape[0])


## 4. GPを再学習し、安全領域内の探索に切り替える

`SafeBOWorkflow` はモデルを自動再学習しません。新しい観測を反映したGPを利用者が渡します。

`mode="expansion"` は安全と予測された点の中で安全モデルの不確実性が高い候補を選びます。**SafeOptの厳密な拡張集合アルゴリズムではありません。**

In [ ]:
updated_safety_model = fit_gp(train_x, train_g)
expansion_candidate = workflow.ask(
    updated_safety_model,
    mode="expansion",
    num_samples=256,
    seed=11,
)
print("探索候補:", expansion_candidate.item())
workflow.tell(expansion_candidate, safety_output(expansion_candidate))


## 5. 運用上の注意

- `beta` が大きいほど一般に安全判定は保守的になります。ただし安全保証を意味しません。
- 初期観測は実際に安全と確認したものを使用してください。
- `ask()` と `tell()` は1回ずつ対応させます。
- 制約違反の実測値もモデル更新に利用してください。
- 候補は有限のSobol点群から選ばれるため、安全な点が存在しても候補集合で見つからない場合があります。
- 実測前の安全性を厳密に保証する仕組みは今回の実装には含まれません。

3手法の比較は `safe_bo_benchmark.ipynb` を参照してください。理論と実装の対応関係はPhase 8のTheory / Guideで整理します。